# Pragmas, Storage and Maintenance

## Quick notes
### What this covers
- How SQLite stores a database in one file: fixed-size **pages**, **B-trees** for tables and indexes, and a **freelist** of unused pages.
- The `PRAGMA` commands you will actually use: settings (`foreign_keys`, `journal_mode`, `synchronous`, `cache_size`, ...), introspection (`table_info`, `index_list`, ...) and checks (`integrity_check`, `foreign_key_check`).
- Seeing where space goes (`dbstat`), reclaiming it (`VACUUM`, `VACUUM INTO`, incremental vacuum), WAL checkpoints, backups, and a health-check query bundle.
### Key points
- A database file is `page_count x page_size` bytes. Deleting rows does **not** shrink the file: freed pages go on the freelist and are reused later. `VACUUM` rebuilds the file compactly.
- Some pragmas are stored **in the file** (`page_size`, `auto_vacuum`, `journal_mode = WAL`); most are **per connection** and must be set again every time you connect (`foreign_keys`, `synchronous`, `cache_size`, `temp_store`).
- `PRAGMA name;` reads a value, `PRAGMA name = value;` sets it. Many pragmas also exist as table-valued functions: `SELECT * FROM pragma_table_info('books')`, which you can join and filter.
- `sqlite_schema` (alias `sqlite_master`) is a table that describes every table, index, view and trigger.
- WAL mode lets readers and one writer work at the same time; a **checkpoint** copies the log back into the main file.
### Gotchas (the part worth revising twice)
- `PRAGMA foreign_keys` is **OFF by default** and resets for every new connection. Constraints you declared are silently not enforced until you turn it on.
- `PRAGMA auto_vacuum` and `page_size` only take effect on a new database (or after `VACUUM`). Setting them on a database that already has tables does nothing at first.
- `VACUUM` cannot run inside a transaction and may need free disk space about the size of the database. It can renumber rowids of tables without an explicit `INTEGER PRIMARY KEY`.
- `synchronous = OFF` or `journal_mode = OFF` can corrupt the database on a crash. Only use them for data you can rebuild.
- `PRAGMA integrity_check` returns the single row `ok` when healthy; anything else is a list of problems. It does **not** check foreign keys; use `foreign_key_check`.
- Two `sqlite3.connect(":memory:")` calls give two **different** empty databases.
### When it matters
- Setting up a connection for an application (foreign keys, WAL, timeouts).
- Database files that keep growing after deletes, or backups and copies of a live database.
- Debugging "why is this file so big?" and checking a file after a crash or a restore.
### One-line recap example
```sql
PRAGMA foreign_keys = ON;                   -- every connection, every time
PRAGMA journal_mode = WAL;                  -- stored in the file
SELECT * FROM pragma_integrity_check;       -- 'ok' when healthy
```

## 01. Seed data: authors, books, a view and a trigger
- A small library schema with every kind of schema object (`reviews` has a foreign key with deliberately no index; section 22 finds it), so that `sqlite_schema` and the introspection pragmas have something to show.
- `books` has about 1500 rows with a long text column, so the data spans many pages.
- A generated column (`price_with_tax`) shows up in `table_xinfo` but not in `table_info`.
- File-based demos later in the notebook use a temporary directory that is created and deleted inside one Python cell; paths are never printed.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE authors (
    author_id INTEGER PRIMARY KEY,
    name      TEXT NOT NULL UNIQUE,
    country   TEXT
);
CREATE TABLE books (
    book_id   INTEGER PRIMARY KEY,
    author_id INTEGER NOT NULL REFERENCES authors (author_id) ON DELETE CASCADE,
    title     TEXT NOT NULL,
    year      INTEGER NOT NULL,
    price     REAL NOT NULL,
    price_with_tax REAL GENERATED ALWAYS AS (round(price * 1.2, 2)) VIRTUAL,
    blurb     TEXT NOT NULL
);
CREATE INDEX idx_books_author ON books (author_id);
CREATE INDEX idx_books_year_price ON books (year, price);
CREATE TABLE book_audit (audit_id INTEGER PRIMARY KEY, book_id INTEGER, note TEXT);
CREATE TABLE reviews (
    review_id INTEGER PRIMARY KEY,
    book_id   INTEGER NOT NULL REFERENCES books (book_id),   -- note: no index on book_id
    stars     INTEGER NOT NULL
);
CREATE VIEW v_book_list AS
    SELECT b.book_id, b.title, a.name AS author FROM books b JOIN authors a USING (author_id);
CREATE TRIGGER trg_books_ai AFTER INSERT ON books
BEGIN
    INSERT INTO book_audit (book_id, note) VALUES (NEW.book_id, 'inserted');
END;

WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 30)
INSERT INTO authors SELECT i, 'Author ' || printf('%02d', i), CASE i % 3 WHEN 0 THEN 'IN' WHEN 1 THEN 'UK' ELSE NULL END FROM n;

WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 1500)
INSERT INTO books (book_id, author_id, title, year, price, blurb)
SELECT i, 1 + i % 30, 'Book ' || printf('%04d', i), 1990 + i % 35, 5 + (i * 7) % 40,
       'Lorem ipsum dolor sit amet ' || i || ' consectetur adipiscing elit sed do eiusmod tempor incididunt ut labore'
FROM n;

SELECT (SELECT COUNT(*) FROM authors) AS authors, (SELECT COUNT(*) FROM books) AS books, (SELECT COUNT(*) FROM book_audit) AS audit_rows;

authors | books | audit_rows
--------+-------+-----------
30      | 1500  | 1500      
(1 row)



## 02. How SQLite stores data: pages, B-trees, freelist
- The whole database is **one file** made of equal-size **pages** (default 4096 bytes). File size = `page_count x page_size`.
- Each table is a **B-tree** of pages keyed by rowid; each index is its own B-tree keyed by the indexed columns. Every B-tree starts at a **root page** (see `rootpage` in `sqlite_schema`).
- Rows that do not fit in a page spill into **overflow pages**. Small rows are packed many to a page.
- When rows are deleted and a page becomes empty, the page goes to the **freelist** and is reused by later inserts. The file does not get smaller by itself.
- Page 1 holds the file header and the schema table. `PRAGMA page_count` includes freelist pages.

In [3]:
%%sql
PRAGMA page_size;
PRAGMA page_count;
PRAGMA freelist_count;

page_size
---------
4096     
(1 row)

page_count
----------
72        
(1 row)

freelist_count
--------------
0             
(1 row)



In [4]:
%%sql
-- the file size an on-disk copy of this database would have
SELECT (SELECT page_count FROM pragma_page_count) * (SELECT page_size FROM pragma_page_size) AS bytes_if_saved,
       (SELECT page_size FROM pragma_page_size) AS page_size;

bytes_if_saved | page_size
---------------+----------
294912         | 4096     
(1 row)



## 03. sqlite_schema (sqlite_master): the catalogue
- `sqlite_schema` (older name `sqlite_master`, both work) has one row per table, index, view and trigger.
- Columns: `type`, `name`, `tbl_name` (the table it belongs to), `rootpage` (first page of its B-tree; 0 for views and triggers) and `sql` (the original `CREATE` text).
- Names starting with `sqlite_` are internal; `sqlite_autoindex_...` entries are indexes SQLite created for `UNIQUE` / `PRIMARY KEY` constraints.
- The `sql` column is the exact text you wrote, so you can rebuild a schema from it.

In [5]:
%%sql
SELECT type, name, tbl_name, rootpage FROM sqlite_schema ORDER BY type, name;

type    | name                       | tbl_name    | rootpage
--------+----------------------------+-------------+---------
index   | idx_books_author           | books       | 5       
index   | idx_books_year_price       | books       | 6       
index   | sqlite_autoindex_authors_1 | authors     | 3       
table   | authors                    | authors     | 2       
table   | book_audit                 | book_audit  | 7       
table   | books                      | books       | 4       
table   | reviews                    | reviews     | 8       
trigger | trg_books_ai               | books       | 0       
view    | v_book_list                | v_book_list | 0       
(9 rows)



In [6]:
%%sql
-- counts per object type, and the stored CREATE text of one object
SELECT type, COUNT(*) AS objects FROM sqlite_schema GROUP BY type ORDER BY type;
SELECT sql FROM sqlite_schema WHERE name = 'v_book_list';

type    | objects
--------+--------
index   | 3      
table   | 4      
trigger | 1      
view    | 1      
(4 rows)

sql                                                                                                                     
------------------------------------------------------------------------------------------------------------------------
CREATE VIEW v_book_list AS
    SELECT b.book_id, b.title, a.name AS author FROM books b JOIN authors a USING (author_id)
(1 row)



## 04. Introspection pragmas: columns, indexes, foreign keys
- `table_info(t)`: one row per column: `cid`, `name`, `type`, `notnull`, `dflt_value`, `pk` (position in the primary key, 0 if none).
- `table_xinfo(t)`: the same plus a `hidden` column. Generated columns appear here (hidden = 2 for virtual, 3 for stored) but **not** in `table_info`.
- `index_list(t)`: indexes on the table (`unique`, `origin`: `c` = CREATE INDEX, `u` = UNIQUE constraint, `pk` = primary key, `partial`).
- `index_info(i)`: the columns of one index in order. `index_xinfo(i)` adds the sort order, collation and whether a column is a key or just the rowid.
- `foreign_key_list(t)`: the foreign keys declared on the table.

In [7]:
%%sql
PRAGMA table_info(books);

cid | name      | type    | notnull | dflt_value | pk
----+-----------+---------+---------+------------+---
0   | book_id   | INTEGER | 0       | NULL       | 1 
1   | author_id | INTEGER | 1       | NULL       | 0 
2   | title     | TEXT    | 1       | NULL       | 0 
3   | year      | INTEGER | 1       | NULL       | 0 
4   | price     | REAL    | 1       | NULL       | 0 
5   | blurb     | TEXT    | 1       | NULL       | 0 
(6 rows)



In [8]:
%%sql
-- the generated column only appears in table_xinfo
SELECT cid, name, type, hidden FROM pragma_table_xinfo('books') ORDER BY cid;

cid | name           | type    | hidden
----+----------------+---------+-------
0   | book_id        | INTEGER | 0     
1   | author_id      | INTEGER | 0     
2   | title          | TEXT    | 0     
3   | year           | INTEGER | 0     
4   | price          | REAL    | 0     
5   | price_with_tax | REAL    | 2     
6   | blurb          | TEXT    | 0     
(7 rows)



In [9]:
%%sql
PRAGMA index_list(books);
PRAGMA index_info(idx_books_year_price);
PRAGMA foreign_key_list(books);

seq | name                 | unique | origin | partial
----+----------------------+--------+--------+--------
0   | idx_books_year_price | 0      | c      | 0      
1   | idx_books_author     | 0      | c      | 0      
(2 rows)

seqno | cid | name 
------+-----+------
0     | 3   | year 
1     | 4   | price
(2 rows)

id | seq | table   | from      | to        | on_update | on_delete | match
---+-----+---------+-----------+-----------+-----------+-----------+------
0  | 0   | authors | author_id | author_id | NO ACTION | CASCADE   | NONE 
(1 row)



## 05. Pragma functions: introspect the whole schema in one query
- A pragma with an argument can be written as a table-valued function: `pragma_table_info('books')`. Unlike `PRAGMA table_info(books)` it can be joined and filtered with normal SQL.
- Join `sqlite_schema` to `pragma_table_info(name)` to list **every column of every table** at once. Pass a column from the left table as the argument.
- Typical uses: find all columns called `email`, list tables without a primary key, or generate documentation.

In [10]:
%%sql
SELECT m.name AS table_name, p.cid, p.name AS column_name, p.type, p."notnull" AS not_null, p.pk
FROM sqlite_schema AS m JOIN pragma_table_info(m.name) AS p
WHERE m.type = 'table' AND m.name NOT LIKE 'sqlite_%'
ORDER BY m.name, p.cid;

table_name | cid | column_name | type    | not_null | pk
-----------+-----+-------------+---------+----------+---
authors    | 0   | author_id   | INTEGER | 0        | 1 
authors    | 1   | name        | TEXT    | 1        | 0 
authors    | 2   | country     | TEXT    | 0        | 0 
book_audit | 0   | audit_id    | INTEGER | 0        | 1 
book_audit | 1   | book_id     | INTEGER | 0        | 0 
book_audit | 2   | note        | TEXT    | 0        | 0 
books      | 0   | book_id     | INTEGER | 0        | 1 
books      | 1   | author_id   | INTEGER | 1        | 0 
books      | 2   | title       | TEXT    | 1        | 0 
books      | 3   | year        | INTEGER | 1        | 0 
books      | 4   | price       | REAL    | 1        | 0 
books      | 5   | blurb       | TEXT    | 1        | 0 
reviews    | 0   | review_id   | INTEGER | 0        | 1 
reviews    | 1   | book_id     | INTEGER | 1        | 0 
reviews    | 2   | stars       | INTEGER | 1        | 0 
(15 rows)



In [11]:
%%sql
-- tables that have no primary key at all (we add one scratch table that has none, so the query has something to find)
CREATE TABLE scratch_log (msg TEXT);
SELECT m.name FROM sqlite_schema m
WHERE m.type = 'table' AND m.name NOT LIKE 'sqlite_%'
  AND NOT EXISTS (SELECT 1 FROM pragma_table_info(m.name) WHERE pk > 0)
ORDER BY m.name;

name       
-----------
scratch_log
(1 row)



In [12]:
%%sql
-- every foreign key in the database, with its delete rule
SELECT m.name AS child_table, f."from" AS child_column, f."table" AS parent_table, f."to" AS parent_column, f.on_delete
FROM sqlite_schema m JOIN pragma_foreign_key_list(m.name) f
WHERE m.type = 'table'
ORDER BY m.name;

child_table | child_column | parent_table | parent_column | on_delete
------------+--------------+--------------+---------------+----------
books       | author_id    | authors      | author_id     | CASCADE  
reviews     | book_id      | books        | book_id       | NO ACTION
(2 rows)



## 06. database_list, sqlite_version and compile options
- `PRAGMA database_list` shows each attached database: `main`, `temp` (once used) and anything you `ATTACH`ed. For an in-memory database the file column is empty.
- `sqlite_version()` is the library version. Features depend on it (`RETURNING` 3.35+, `STRICT` 3.37+, `->>` JSON operator 3.38+).
- `PRAGMA compile_options` lists the options the library was built with. `sqlite_compileoption_used('NAME')` answers 1 or 0. Use it to check for optional features such as FTS5 or the `dbstat` table before relying on them.
- Results depend on how your Python's SQLite was built, so treat the values below as examples.

In [13]:
%%sql
PRAGMA database_list;

seq | name | file
----+------+-----
0   | main |     
(1 row)



In [14]:
%%sql
SELECT sqlite_version() >= '3.45' AS version_at_least_3_45,
       sqlite_compileoption_used('ENABLE_FTS5') AS has_fts5,
       sqlite_compileoption_used('ENABLE_DBSTAT_VTAB') AS has_dbstat,
       (SELECT COUNT(*) FROM pragma_compile_options) > 10 AS many_options_listed;

version_at_least_3_45 | has_fts5 | has_dbstat | many_options_listed
----------------------+----------+------------+--------------------
1                     | 1        | 1          | 1                  
(1 row)



## 07. Limits: the SQLITE_MAX_* knobs
- SQLite has built-in limits: maximum length of a string or blob, number of columns, SQL text length, expression depth, bound parameters, attached databases, and so on. They are set when the library is compiled.
- Python reads them with `con.getlimit(sqlite3.SQLITE_LIMIT_...)` and can **lower** them for one connection with `setlimit`. Lowering limits is a defensive tool when you run untrusted SQL.
- Typical defaults: 2000 columns per table, 1,000,000,000 bytes per value, 32766 or more bound parameters (999 in old versions), 10 attached databases, 1000 trigger depth.
- Exceeding a limit raises an error such as `too many columns` or `string or blob too big`.

In [15]:
if hasattr(sqlite3.Connection, 'getlimit'):
    import sqlite3
    names = ["LENGTH", "SQL_LENGTH", "COLUMN", "EXPR_DEPTH", "COMPOUND_SELECT", "FUNCTION_ARG", "ATTACHED",
             "LIKE_PATTERN_LENGTH", "VARIABLE_NUMBER", "TRIGGER_DEPTH"]
    for n in names:
        print(f"SQLITE_LIMIT_{n:<20}", con.getlimit(getattr(sqlite3, "SQLITE_LIMIT_" + n)))
else:
    print('SKIP: SQLite runtime limit inspection requires Python 3.11+')

SQLITE_LIMIT_LENGTH               1000000000
SQLITE_LIMIT_SQL_LENGTH           1000000000
SQLITE_LIMIT_COLUMN               2000
SQLITE_LIMIT_EXPR_DEPTH           1000
SQLITE_LIMIT_COMPOUND_SELECT      500
SQLITE_LIMIT_FUNCTION_ARG         1000
SQLITE_LIMIT_ATTACHED             10
SQLITE_LIMIT_LIKE_PATTERN_LENGTH  50000
SQLITE_LIMIT_VARIABLE_NUMBER      250000
SQLITE_LIMIT_TRIGGER_DEPTH        1000


In [16]:
if hasattr(sqlite3.Connection, 'getlimit'):
    # lower a limit on a throw-away connection and hit it
    tmp_con = sqlite3.connect(":memory:")
    tmp_con.setlimit(sqlite3.SQLITE_LIMIT_COLUMN, 6)   # sqlite_schema itself has 5 columns, so 6 is about the lowest usable value
    try:
        tmp_con.execute("CREATE TABLE wide (a, b, c, d, e, f, g)")
    except sqlite3.Error as exc:
        print("error:", exc)
    tmp_con.execute("CREATE TABLE ok (a, b, c, d, e, f)")
    print("6 columns are fine; limit now:", tmp_con.getlimit(sqlite3.SQLITE_LIMIT_COLUMN))
    tmp_con.close()
else:
    print('SKIP: SQLite runtime limit inspection requires Python 3.11+')

error: too many columns on wide
6 columns are fine; limit now: 6


## 08. Pragmas: what is stored in the file and what is per connection
- **Stored in the database file** (persist after you disconnect): `page_size`, `auto_vacuum`, `application_id`, `user_version`, and `journal_mode = WAL`.
- **Per connection** (must be set again on every connect): `foreign_keys`, `synchronous`, `cache_size`, `temp_store`, `mmap_size`, `busy_timeout`, `query_only`, `trusted_schema` and the rollback journal modes other than WAL.
- Rule of thumb: put the connection-level pragmas in the one function that opens your connection, so no code path forgets them.
- `user_version` and `application_id` are free integers you can use for schema-migration numbers and file-type tagging.

In [17]:
%%sql
PRAGMA user_version = 3;
PRAGMA user_version;

user_version
------------
3           
(1 row)



## 09. foreign_keys: declared is not enforced
- `foreign_keys` is OFF by default in raw SQLite for backwards compatibility. A `REFERENCES` clause does nothing until each connection runs `PRAGMA foreign_keys = ON`.
- It cannot be changed inside a transaction (the command is silently ignored there).
- This notebook's connection turns it ON at the start. Below we show the error, then turn it OFF to show that the orphan row is then accepted, and finally find the orphan with `foreign_key_check`.
- See `05-DDL-And-Integrity` for foreign key design and `ON DELETE` rules.

In [18]:
%%sql expect_error
PRAGMA foreign_keys;
INSERT INTO books (book_id, author_id, title, year, price, blurb) VALUES (9001, 999, 'Orphan', 2000, 10, 'x');

foreign_keys
------------
1           
(1 row)

ERROR: FOREIGN KEY constraint failed



In [19]:
%%sql
PRAGMA foreign_keys = OFF;
INSERT INTO books (book_id, author_id, title, year, price, blurb) VALUES (9001, 999, 'Orphan', 2000, 10, 'x');
PRAGMA foreign_keys;

foreign_keys
------------
0           
(1 row)



In [20]:
%%sql
-- foreign_key_check lists violations: table, rowid, parent table, index of the FK
SELECT "table", rowid, parent FROM pragma_foreign_key_check ORDER BY rowid;

table | rowid | parent 
------+-------+--------
books | 9001  | authors
(1 row)



In [21]:
%%sql
DELETE FROM books WHERE book_id = 9001;
DELETE FROM book_audit WHERE book_id = 9001;
PRAGMA foreign_keys = ON;
SELECT COUNT(*) AS violations FROM pragma_foreign_key_check;

violations
----------
0         
(1 row)



## 10. cache_size, temp_store, mmap_size, synchronous
- `cache_size`: how many pages SQLite keeps in memory. A positive number is pages; a **negative** number is kibibytes (`-2000` = about 2 MB, the default). Bigger caches help big working sets.
- `temp_store`: where temporary tables, sort B-trees and indexes live. `0` = compile-time default, `1` = file, `2` = memory. Use `MEMORY` if you sort large results and have RAM.
- `mmap_size`: bytes of the file to read through memory mapping (0 = off). Can speed up reads on 64-bit systems. An in-memory database has nothing to map, so `PRAGMA mmap_size` returns no row there (try it on a file database: `PRAGMA mmap_size = 268435456`).
- `synchronous` (how hard SQLite waits for the disk): `0` OFF, `1` NORMAL, `2` FULL, `3` EXTRA. FULL is the safe default. With WAL, NORMAL is a common safe-enough choice: a crash can lose the last commits but cannot corrupt the file.
- `busy_timeout` (milliseconds) makes a connection wait for a lock instead of failing at once with `database is locked`.

In [22]:
%%sql
PRAGMA cache_size;
PRAGMA temp_store;
PRAGMA synchronous;

cache_size
----------
-2000     
(1 row)

temp_store
----------
0         
(1 row)

synchronous
-----------
2          
(1 row)



In [23]:
%%sql
-- (PRAGMA busy_timeout = N also echoes the new value back as a row, so the first table below is that echo)
PRAGMA cache_size = -8000;
PRAGMA temp_store = MEMORY;
PRAGMA busy_timeout = 2000;
SELECT * FROM pragma_cache_size;
SELECT * FROM pragma_temp_store;
SELECT * FROM pragma_busy_timeout;

timeout
-------
2000   
(1 row)

cache_size
----------
-8000     
(1 row)

temp_store
----------
2         
(1 row)

timeout
-------
2000   
(1 row)



## 11. dbstat: where does the space go?
- `dbstat` is a virtual table with one row per page (when the library was built with it; section 06 showed how to check). Each row says which B-tree (table or index `name`) owns the page, how many `cell`s (rows or entries) it holds, how many bytes of `payload`, and how many bytes are `unused` on the page.
- Group by `name` to see the pages used by each table and index. This answers "why is my file big?" and "what does that index cost?".
- `pgsize` is the page size; `pageno` is the page number. The `aggregate = TRUE` form of dbstat gives one row per B-tree directly.

In [24]:
%%sql
SELECT name, COUNT(*) AS pages, SUM(ncell) AS cells, SUM(payload) AS payload_bytes, SUM(unused) AS unused_bytes
FROM dbstat
GROUP BY name
ORDER BY pages DESC, name;

name                       | pages | cells | payload_bytes | unused_bytes
---------------------------+-------+-------+---------------+-------------
books                      | 49    | 1547  | 183343        | 9219        
book_audit                 | 8     | 1506  | 20872         | 4407        
idx_books_year_price       | 6     | 1500  | 13372         | 6636        
idx_books_author           | 5     | 1500  | 8822          | 7102        
authors                    | 1     | 30    | 430           | 3538        
reviews                    | 1     | 0     | 0             | 4088        
scratch_log                | 1     | 0     | 0             | 4088        
sqlite_autoindex_authors_1 | 1     | 30    | 389           | 3609        
sqlite_schema              | 1     | 10    | 1429          | 2514        
(9 rows)



In [25]:
%%sql
-- share of the file per object, as a percentage of all pages
SELECT name, COUNT(*) AS pages,
       round(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dbstat), 1) AS pct_of_pages
FROM dbstat
GROUP BY name
HAVING pct_of_pages >= 1
ORDER BY pages DESC, name;

name                       | pages | pct_of_pages
---------------------------+-------+-------------
books                      | 49    | 67.1        
book_audit                 | 8     | 11.0        
idx_books_year_price       | 6     | 8.2         
idx_books_author           | 5     | 6.8         
authors                    | 1     | 1.4         
reviews                    | 1     | 1.4         
scratch_log                | 1     | 1.4         
sqlite_autoindex_authors_1 | 1     | 1.4         
sqlite_schema              | 1     | 1.4         
(9 rows)



## 12. What deletes do to file size (and the freelist)
- We make a real file in a temporary directory, load 3000 rows, delete 80 percent, and watch `page_count` and `freelist_count`.
- After the delete the **page count stays the same**; the freed pages are counted in `freelist_count`. New inserts would reuse them.
- `VACUUM` rewrites the database without the free pages, so `page_count` drops and `freelist_count` becomes 0.
- Sizes are shown in pages (and pages x page size) rather than file paths or timings, so the numbers are stable.

In [26]:
import os, tempfile

def pages(c):
    return (c.execute("PRAGMA page_count").fetchone()[0], c.execute("PRAGMA freelist_count").fetchone()[0])

with tempfile.TemporaryDirectory() as tmp:
    c = sqlite3.connect(os.path.join(tmp, "demo.db"), isolation_level=None)
    c.execute("CREATE TABLE log (id INTEGER PRIMARY KEY, msg TEXT NOT NULL)")
    c.execute("""WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 3000)
                 INSERT INTO log SELECT i, 'message number ' || i || ' padded with some text to take up room' FROM n""")
    print("after load   : (page_count, freelist) =", pages(c))
    c.execute("DELETE FROM log WHERE id % 5 != 0")
    print("after delete : (page_count, freelist) =", pages(c), "| rows left:", c.execute("SELECT COUNT(*) FROM log").fetchone()[0])
    c.execute("INSERT INTO log SELECT id + 10000, msg FROM log WHERE id <= 500")
    print("after reinsert (pages reused):", pages(c))
    c.execute("DELETE FROM log WHERE id > 10000")
    c.execute("VACUUM")
    print("after VACUUM : (page_count, freelist) =", pages(c))
    c.close()

after load   : (page_count, freelist) = (51, 0)
after delete : (page_count, freelist) = (51, 37) | rows left: 600
after reinsert (pages reused): (51, 35)
after VACUUM : (page_count, freelist) = (12, 0)


## 13. VACUUM and VACUUM INTO
- `VACUUM` rebuilds the entire database into a temporary copy and then replaces the original. It removes free pages, defragments tables and indexes, and can change the stored page size or auto_vacuum mode.
- It needs free disk space up to about the size of the database, takes an exclusive lock, and **cannot run inside a transaction**.
- `VACUUM INTO 'file'` writes the compacted copy to a **new file** and leaves the original untouched. It is a handy way to take a clean snapshot or a backup of a live database.
- Tables without an explicit `INTEGER PRIMARY KEY` may get their hidden rowids renumbered by `VACUUM`; do not store rowids outside the database.

In [27]:
with tempfile.TemporaryDirectory() as tmp:
    src = sqlite3.connect(os.path.join(tmp, "src.db"), isolation_level=None)
    src.execute("CREATE TABLE log (id INTEGER PRIMARY KEY, msg TEXT NOT NULL)")
    src.execute("""WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 3000)
                   INSERT INTO log SELECT i, 'message number ' || i || ' padded with some text to take up room' FROM n""")
    src.execute("DELETE FROM log WHERE id % 5 != 0")
    print("original (page_count, freelist):", pages(src))
    src.execute("VACUUM INTO ?", (os.path.join(tmp, "copy.db"),))
    print("original untouched          :", pages(src))
    copy = sqlite3.connect(os.path.join(tmp, "copy.db"))
    print("compact copy (page_count, freelist):", pages(copy))
    print("copy has the same rows      :", copy.execute("SELECT COUNT(*) FROM log").fetchone()[0] == src.execute("SELECT COUNT(*) FROM log").fetchone()[0])
    print("copy integrity_check        :", copy.execute("PRAGMA integrity_check").fetchone()[0])
    copy.close(); src.close()

original (page_count, freelist): (51, 37)
original untouched          : (51, 37)
compact copy (page_count, freelist): (12, 0)
copy has the same rows      : True
copy integrity_check        : ok


## 14. auto_vacuum and incremental vacuum
- `PRAGMA auto_vacuum` has three modes: `0` NONE (default, file never shrinks by itself), `1` FULL (the file shrinks at every commit that frees pages), `2` INCREMENTAL (free pages are tracked; you release them with `PRAGMA incremental_vacuum(N)` when you choose).
- It must be set **before the first table is created** (or followed by a `VACUUM`). The setting is stored in the file.
- FULL and INCREMENTAL keep extra bookkeeping pages (pointer maps), so the file is slightly larger and writes do a bit more work.
- INCREMENTAL is a good fit for databases with big periodic deletes: reclaim space in small steps without the long lock that a full `VACUUM` needs.

In [28]:
with tempfile.TemporaryDirectory() as tmp:
    c = sqlite3.connect(os.path.join(tmp, "inc.db"), isolation_level=None)
    c.execute("PRAGMA auto_vacuum = INCREMENTAL")          # before any table exists
    c.execute("CREATE TABLE log (id INTEGER PRIMARY KEY, msg TEXT NOT NULL)")
    c.execute("""WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 3000)
                 INSERT INTO log SELECT i, 'message number ' || i || ' padded with some text to take up room' FROM n""")
    print("auto_vacuum mode:", c.execute("PRAGMA auto_vacuum").fetchone()[0], "(2 = INCREMENTAL)")
    print("loaded       :", pages(c))
    c.execute("DELETE FROM log WHERE id % 5 != 0")
    print("after delete :", pages(c))
    c.execute("PRAGMA incremental_vacuum(20)").fetchall()  # release 20 free pages (fetchall runs every step)
    print("after incremental_vacuum(20):", pages(c))
    c.execute("PRAGMA incremental_vacuum").fetchall()      # no number = release all free pages
    print("after incremental_vacuum    :", pages(c))
    c.close()

auto_vacuum mode: 2 (2 = INCREMENTAL)
loaded       : (52, 0)
after delete : (52, 37)
after incremental_vacuum(20): (32, 17)
after incremental_vacuum    : (15, 0)


## 15. Journal modes and WAL checkpoints
- **Rollback-journal modes** (`DELETE` default, `TRUNCATE`, `PERSIST`, `MEMORY`, `OFF`): before changing a page SQLite saves the old version in a journal so a crash can be undone. Readers and the writer block each other.
- **WAL mode** (`PRAGMA journal_mode = WAL`): changes are appended to a separate write-ahead log file. Readers keep reading the old data from the main file plus the log, so **readers do not block the writer and the writer does not block readers** (still only one writer at a time). The mode is stored in the file.
- A **checkpoint** copies committed pages from the WAL back into the main file. SQLite does this automatically when the log reaches about 1000 pages (`wal_autocheckpoint`); you can force one with `PRAGMA wal_checkpoint(...)`.
- `wal_checkpoint` returns three numbers: `busy` (1 if it could not finish), frames in the log, frames already copied. Modes: `PASSIVE` (do what is possible without waiting), `FULL`, `RESTART`, `TRUNCATE` (also empties the log file).
- WAL does not work on some network file systems, and the database is then 3 files (`-wal` and `-shm` beside it): copy all of them, or use the backup API (section 16).

In [29]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "wal.db")
    w = sqlite3.connect(path, isolation_level=None)
    print("default journal_mode :", w.execute("PRAGMA journal_mode").fetchone()[0])
    print("switch to WAL        :", w.execute("PRAGMA journal_mode = WAL").fetchone()[0])
    print("wal_autocheckpoint   :", w.execute("PRAGMA wal_autocheckpoint").fetchone()[0], "pages")
    w.execute("PRAGMA wal_autocheckpoint = 0")              # we will checkpoint by hand
    w.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, v TEXT)")
    w.execute("INSERT INTO t (v) VALUES ('first')")
    print("checkpoint(PASSIVE)  : (busy, frames_in_log, frames_checkpointed) =", w.execute("PRAGMA wal_checkpoint(PASSIVE)").fetchone())
    w.execute("INSERT INTO t (v) VALUES ('second')")
    print("one more commit      : (busy, frames, checkpointed) =", w.execute("PRAGMA wal_checkpoint(PASSIVE)").fetchone())
    print("checkpoint(TRUNCATE) :", w.execute("PRAGMA wal_checkpoint(TRUNCATE)").fetchone(), "-> log emptied")

    # readers and the writer work at the same time
    r = sqlite3.connect(path, isolation_level=None)
    r.execute("BEGIN")
    print("reader sees rows     :", r.execute("SELECT COUNT(*) FROM t").fetchone()[0])
    w.execute("INSERT INTO t (v) VALUES ('third')")         # not blocked by the open reader
    print("reader still sees    :", r.execute("SELECT COUNT(*) FROM t").fetchone()[0], "(old snapshot)")
    r.execute("COMMIT")
    print("after reader commits :", r.execute("SELECT COUNT(*) FROM t").fetchone()[0])
    r.close(); w.close()

default journal_mode : delete
switch to WAL        : wal
wal_autocheckpoint   : 1000 pages
checkpoint(PASSIVE)  : (busy, frames_in_log, frames_checkpointed) = (0, 3, 3)
one more commit      : (busy, frames, checkpointed) = (0, 1, 1)
checkpoint(TRUNCATE) : (0, 0, 0) -> log emptied
reader sees rows     : 2
reader still sees    : 2 (old snapshot)
after reader commits : 3


In [30]:
# the rollback-journal modes: the pragma returns the mode now in force
with tempfile.TemporaryDirectory() as tmp:
    c = sqlite3.connect(os.path.join(tmp, "modes.db"), isolation_level=None)
    for mode in ["DELETE", "TRUNCATE", "PERSIST", "MEMORY", "OFF", "WAL", "DELETE"]:
        print(f"journal_mode = {mode:<8} ->", c.execute(f"PRAGMA journal_mode = {mode}").fetchone()[0])
    c.close()

journal_mode = DELETE   -> delete
journal_mode = TRUNCATE -> truncate
journal_mode = PERSIST  -> persist
journal_mode = MEMORY   -> memory
journal_mode = OFF      -> off
journal_mode = WAL      -> wal
journal_mode = DELETE   -> delete


## 16. Backups with Connection.backup
- Copying a database file while someone writes to it can give a broken copy. The **backup API** (`source.backup(target)` in Python) copies a consistent snapshot, even while the source is in use.
- The target can be a file or an in-memory database, and the source can too. Great for tests (load a file into memory) and for nightly snapshots.
- `pages=N` copies N pages per step and calls an optional `progress(status, remaining, total)` function between steps, so you can show progress or let other writers in.
- `VACUUM INTO` (section 13) is the SQL-only alternative: it also gives a consistent, compacted copy.

In [31]:
with tempfile.TemporaryDirectory() as tmp:
    disk = sqlite3.connect(os.path.join(tmp, "disk.db"), isolation_level=None)
    disk.execute("CREATE TABLE log (id INTEGER PRIMARY KEY, msg TEXT NOT NULL)")
    disk.execute("""WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 3000)
                    INSERT INTO log SELECT i, 'message number ' || i || ' padded with some text to take up room' FROM n""")
    total_pages = disk.execute("PRAGMA page_count").fetchone()[0]
    print("source pages:", total_pages)

    mem = sqlite3.connect(":memory:")
    seen = []
    disk.backup(mem, pages=20, progress=lambda status, remaining, total: seen.append(remaining))
    print("progress calls (remaining pages after each step):", seen)
    print("rows in the in-memory copy:", mem.execute("SELECT COUNT(*) FROM log").fetchone()[0])

    # and the other way: save a memory database to a file
    out = sqlite3.connect(os.path.join(tmp, "saved.db"))
    mem.backup(out)
    print("rows in the saved file    :", out.execute("SELECT COUNT(*) FROM log").fetchone()[0])
    out.close(); mem.close(); disk.close()

source pages: 51
progress calls (remaining pages after each step): [31, 11, 0]
rows in the in-memory copy: 3000
rows in the saved file    : 3000


## 17. In-memory versus file databases
- `:memory:` is a database that lives in RAM and disappears when the connection closes. Fast and ideal for tests, scratch work and learning (like this notebook).
- **Every `sqlite3.connect(":memory:")` is its own empty database.** To share one between connections in the same process use a shared-cache URI such as `file:name?mode=memory&cache=shared` with `uri=True`; it lives while at least one connection is open.
- A file database survives restarts, can be backed up and copied, and supports WAL. It is slower because commits wait for the disk (`synchronous`).
- The `temp` database (`CREATE TEMP TABLE ...`) is private to a connection and removed when it closes; where it is stored is controlled by `temp_store`.

In [32]:
a = sqlite3.connect(":memory:"); b = sqlite3.connect(":memory:")
a.execute("CREATE TABLE only_in_a (x)")
print("tables seen by a:", a.execute("SELECT COUNT(*) FROM sqlite_schema").fetchone()[0],
      "| tables seen by b:", b.execute("SELECT COUNT(*) FROM sqlite_schema").fetchone()[0], "(separate databases)")
s1 = sqlite3.connect("file:pragma_demo_mem?mode=memory&cache=shared", uri=True)
s2 = sqlite3.connect("file:pragma_demo_mem?mode=memory&cache=shared", uri=True)
s1.execute("CREATE TABLE shared_t (x)"); s1.execute("INSERT INTO shared_t VALUES (1)"); s1.commit()
print("second shared connection sees:", s2.execute("SELECT COUNT(*) FROM shared_t").fetchone()[0], "row")
for x in (a, b, s1, s2): x.close()

tables seen by a: 1 | tables seen by b: 0 (separate databases)
second shared connection sees: 1 row


## 18. STRICT and WITHOUT ROWID: effect on size
- `WITHOUT ROWID` stores the table **as the primary-key index itself**. For a table with a non-integer primary key (like a text code) a normal rowid table stores the key twice: once in the table row and once in the automatic unique index on the key. WITHOUT ROWID stores it once.
- It helps most when rows are small and the key is a text or composite key. It is a poor fit for tables with very wide rows.
- `STRICT` (3.37+) only enforces column types. It does **not** change how rows are stored, so the size is the same.
- Below, three tables hold the same 3000 rows. `dbstat` counts the pages used by each table plus its auto-index.

In [33]:
%%sql
CREATE TABLE codes_plain  (code TEXT PRIMARY KEY, label TEXT NOT NULL, qty INTEGER NOT NULL);
CREATE TABLE codes_strict (code TEXT PRIMARY KEY, label TEXT NOT NULL, qty INTEGER NOT NULL) STRICT;
CREATE TABLE codes_norowid (code TEXT PRIMARY KEY, label TEXT NOT NULL, qty INTEGER NOT NULL) WITHOUT ROWID;

WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 3000)
INSERT INTO codes_plain SELECT 'CODE-' || printf('%05d', i), 'label ' || i, i % 100 FROM n;
INSERT INTO codes_strict SELECT * FROM codes_plain;
INSERT INTO codes_norowid SELECT * FROM codes_plain;

SELECT CASE WHEN name LIKE 'sqlite_autoindex_codes_plain%' THEN 'codes_plain (+ auto-index)'
            WHEN name LIKE 'sqlite_autoindex_codes_strict%' THEN 'codes_strict (+ auto-index)'
            ELSE name END AS storage,
       COUNT(*) AS pages
FROM dbstat WHERE name LIKE 'codes_%' OR name LIKE 'sqlite_autoindex_codes_%'
GROUP BY storage ORDER BY storage;

storage                     | pages
----------------------------+------
codes_norowid               | 24   
codes_plain                 | 23   
codes_plain (+ auto-index)  | 16   
codes_strict                | 23   
codes_strict (+ auto-index) | 16   
(5 rows)



In [34]:
%%sql
-- same pages per object type, summed per table (table B-tree + its automatic index)
SELECT t.name AS tbl,
       (SELECT COUNT(*) FROM dbstat d WHERE d.name = t.name
           OR d.name IN (SELECT i.name FROM pragma_index_list(t.name) i WHERE i.origin = 'pk')) AS total_pages
FROM sqlite_schema t WHERE t.type = 'table' AND t.name LIKE 'codes_%' ORDER BY t.name;

tbl           | total_pages
--------------+------------
codes_norowid | 24         
codes_plain   | 39         
codes_strict  | 39         
(3 rows)



## 19. Defensive and security pragmas
- Run untrusted SQL (user-supplied queries, files from unknown sources) with the smallest powers possible.
- `PRAGMA query_only = ON`: this connection cannot write at all.
- `PRAGMA trusted_schema = OFF`: functions called from views, triggers, `CHECK` constraints and generated columns may only be "safe" functions. Protects you from a crafted database file whose schema calls dangerous functions. Recommended when opening files you did not create.
- **Defensive mode** (C API `SQLITE_DBCONFIG_DEFENSIVE`; in Python 3.12+ `con.setconfig(sqlite3.SQLITE_DBCONFIG_DEFENSIVE, True)`): blocks features that can corrupt the file on purpose, such as writing to `sqlite_schema` through `PRAGMA writable_schema`.
- Also consider `PRAGMA secure_delete = ON` (overwrite deleted content with zeros so it does not linger in free pages), `PRAGMA cell_size_check = ON`, `setlimit` (section 07) and `PRAGMA foreign_keys = ON`.
- `PRAGMA ignore_check_constraints` and `writable_schema` are the opposite: debug-only switches that bypass safety. Do not leave them on.

In [35]:
safe = sqlite3.connect(":memory:", isolation_level=None)
safe.execute("CREATE TABLE t (a)"); safe.execute("INSERT INTO t VALUES (1)")
safe.execute("PRAGMA trusted_schema = OFF")
print("trusted_schema:", safe.execute("PRAGMA trusted_schema").fetchone()[0])
if hasattr(safe, 'setconfig') and hasattr(sqlite3, 'SQLITE_DBCONFIG_DEFENSIVE'):
    safe.setconfig(sqlite3.SQLITE_DBCONFIG_DEFENSIVE, True)
    safe.execute("PRAGMA writable_schema = ON")
    try:
        safe.execute("UPDATE sqlite_schema SET name = 'hacked' WHERE name = 't'")
    except sqlite3.Error as exc:
        print("defensive mode blocks it:", exc)
else:
    print('SKIP: defensive connection configuration requires Python 3.12+ and SQLite support')
safe.execute("PRAGMA query_only = ON")
try:
    safe.execute("INSERT INTO t VALUES (2)")
except sqlite3.Error as exc:
    print("query_only blocks writes:", exc)
safe.close()

trusted_schema: 0
defensive mode blocks it: table sqlite_master may not be modified
query_only blocks writes: attempt to write a readonly database


## 20. PRAGMA optimize and ANALYZE
- `ANALYZE` collects statistics about tables and indexes into `sqlite_stat1` so the planner can choose better plans (see `07-Performance/02-Query-Optimization-Patterns` for a plan that flips).
- `PRAGMA optimize` is the lightweight way: SQLite decides which tables would benefit and analyzes only those. Run it just before you close a long-lived connection, or periodically.
- Bit flags let you look first: `PRAGMA optimize(0x10001)` (check all tables, debug mode) lists what it would do without doing it, one text row per `ANALYZE` it would run. It often returns **no rows**: SQLite only analyzes when it sees a reason (a missing or stale statistic for an index your queries use).
- After large bulk loads or deletes, run `ANALYZE` yourself.

In [36]:
%%sql
PRAGMA optimize(0x10001);

optimize
--------
(0 rows)



In [37]:
%%sql
ANALYZE;
SELECT tbl, idx, stat FROM sqlite_stat1 WHERE tbl IN ('books', 'authors') ORDER BY tbl, idx;

tbl     | idx                        | stat     
--------+----------------------------+----------
authors | sqlite_autoindex_authors_1 | 30 1     
books   | idx_books_author           | 1500 50  
books   | idx_books_year_price       | 1500 43 6
(3 rows)



In [38]:
%%sql
PRAGMA optimize;

optimize
--------
(0 rows)



## 21. integrity_check, quick_check, foreign_key_check
- `PRAGMA integrity_check` walks every page and checks B-tree structure, index-versus-table agreement and constraint columns. It returns one row `ok`, or up to 100 (adjustable: `integrity_check(10)`) descriptions of problems.
- `PRAGMA quick_check` is the same without verifying that indexes match their tables, so it is faster. Use it for routine checks and `integrity_check` after a crash or restore.
- `PRAGMA foreign_key_check` lists rows that break a declared foreign key. It works even when `foreign_keys` is OFF, so it is the right tool for auditing an old database.
- Run these on a **copy** of a suspect file; if something is wrong, restore from backup instead of trusting the file.

In [39]:
%%sql
PRAGMA integrity_check;
PRAGMA quick_check;
PRAGMA integrity_check(5);

integrity_check
---------------
ok             
(1 row)

quick_check
-----------
ok         
(1 row)

integrity_check
---------------
ok             
(1 row)



In [40]:
%%sql
-- table-valued forms return normal rows, so a violation count is one query away
SELECT (SELECT COUNT(*) FROM pragma_integrity_check WHERE integrity_check != 'ok') AS integrity_problems,
       (SELECT COUNT(*) FROM pragma_foreign_key_check) AS fk_violations;

integrity_problems | fk_violations
-------------------+--------------
0                  | 0            
(1 row)



## 22. A database health-check bundle
- One query that reports the key facts about a database, using only pragma functions and `dbstat`: size, wasted space, check results and object counts.
- Run it from a monitoring script or after a restore. `status` is `OK` or `LOOK` so problems are easy to spot.
- Free pages above roughly 20 to 30 percent of the file suggest a `VACUUM` (or an incremental vacuum) would help; the threshold is a judgement call.

In [41]:
%%sql
SELECT check_name, value, status FROM (
    SELECT 1 AS ord, 'integrity_check' AS check_name, (SELECT integrity_check FROM pragma_integrity_check LIMIT 1) AS value,
           CASE WHEN (SELECT integrity_check FROM pragma_integrity_check LIMIT 1) = 'ok' THEN 'OK' ELSE 'LOOK' END AS status
    UNION ALL
    SELECT 2, 'foreign_key_violations', CAST((SELECT COUNT(*) FROM pragma_foreign_key_check) AS TEXT),
           CASE WHEN (SELECT COUNT(*) FROM pragma_foreign_key_check) = 0 THEN 'OK' ELSE 'LOOK' END
    UNION ALL
    SELECT 3, 'pages (total / free)', (SELECT page_count FROM pragma_page_count) || ' / ' || (SELECT freelist_count FROM pragma_freelist_count),
           CASE WHEN (SELECT freelist_count FROM pragma_freelist_count) * 100.0 / (SELECT page_count FROM pragma_page_count) < 25 THEN 'OK' ELSE 'LOOK' END
    UNION ALL
    SELECT 4, 'foreign_keys pragma', CAST((SELECT foreign_keys FROM pragma_foreign_keys) AS TEXT),
           CASE WHEN (SELECT foreign_keys FROM pragma_foreign_keys) = 1 THEN 'OK' ELSE 'LOOK' END
    UNION ALL
    SELECT 5, 'tables / indexes / views / triggers',
           (SELECT COUNT(*) FROM sqlite_schema WHERE type = 'table' AND name NOT LIKE 'sqlite_%') || ' / ' ||
           (SELECT COUNT(*) FROM sqlite_schema WHERE type = 'index') || ' / ' ||
           (SELECT COUNT(*) FROM sqlite_schema WHERE type = 'view') || ' / ' ||
           (SELECT COUNT(*) FROM sqlite_schema WHERE type = 'trigger'), 'OK'
    UNION ALL
    SELECT 6, 'largest object (by pages)', (SELECT name || ' = ' || COUNT(*) FROM dbstat GROUP BY name ORDER BY COUNT(*) DESC, name LIMIT 1), 'OK'
) ORDER BY ord;

check_name                          | value         | status
------------------------------------+---------------+-------
integrity_check                     | ok            | OK    
foreign_key_violations              | 0             | OK    
pages (total / free)                | 180 / 0       | OK    
foreign_keys pragma                 | 1             | OK    
tables / indexes / views / triggers | 8 / 5 / 1 / 1 | OK    
largest object (by pages)           | books = 49    | OK    
(6 rows)



In [42]:
%%sql
-- tables with no index on a foreign key column are a common hidden cost: list the foreign keys that no index covers
SELECT m.name AS child_table, f."from" AS fk_column
FROM sqlite_schema m JOIN pragma_foreign_key_list(m.name) f
WHERE m.type = 'table'
  AND NOT EXISTS (SELECT 1 FROM pragma_index_list(m.name) il JOIN pragma_index_info(il.name) ii
                  WHERE ii.seqno = 0 AND ii.name = f."from")
ORDER BY m.name, f."from";

child_table | fk_column
------------+----------
reviews     | book_id  
(1 row)

